In [49]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [50]:
df = pd.read_csv('dataset/googleplaystore_cleaned.csv')
df

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19456.0,10000,Free,0.0,Everyone
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14336.0,500000,Free,0.0,Everyone
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8908.8,5000000,Free,0.0,Everyone
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25600.0,50000000,Free,0.0,Teen
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2867.2,100000,Free,0.0,Everyone
...,...,...,...,...,...,...,...,...,...
9654,Sya9a Maroc - FR,FAMILY,4.5,38,54272.0,5000,Free,0.0,Everyone
9655,Fr. Mike Schmitz Audio Teachings,FAMILY,5.0,4,3686.4,100,Free,0.0,Everyone
9656,Parkinson Exercices FR,MEDICAL,5.0,3,9728.0,1000,Free,0.0,Everyone
9657,The SCP Foundation DB fr nn5n,BOOKS_AND_REFERENCE,4.5,114,Varies with device,1000,Free,0.0,Mature 17+


### Machine Learning

#### Dataset Overview
The dataset is from 2019 google play store. It contains 10841 rows and 13 columns. The columns are:
1. App: The name of the application.
2. Category: The category of the application.
3. Rating: The average user rating of the application.
4. Reviews: The number of user reviews for the application.
5. Size: The size of the application.
6. Installs: The number of times the application has been installed.
7. Type: The type of the application (Free or Paid).
8. Price: The price of the application.
9. Content Rating: The content rating of the application.
10. Genres: The genres of the application.
11. Last Updated: The date when the application was last updated.
12. Current Ver: The current version of the application.
13. Android Ver: The minimum Android version required to run the application.

#### Data Cleaning
The dataset is so dirty that it contains many missing values, inconsistent data types, and outliers.

#### Columns Included in Cleaned Dataset
1. App
2. Category
3. Rating
4. Reviews
5. Installs
6. Type
7. Price
8. Content Rating

#### Questions to Answer
1. Can we accurately predict if an application will achieve widespread popularity based on its pre-launch metadata, category, and monetization strategy?


### Objective 1

In [51]:
df['Success'] = (df['Installs'] > 100_000).astype(int)  # Make it binary
features = ['Category','Size', 'Type', 'Price', 'Content Rating']

X = df[features]
y = df['Success']

In [52]:
X_encoded = pd.get_dummies(X, columns=['Category', 'Type', 'Content Rating', 'Size'], drop_first=True)

In [53]:
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=67)

In [54]:
# Random Forest
print("Planting Trees...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=67, n_jobs=-1)

rf_model.fit(X_train, y_train)
print("Predicting...")
y_pred = rf_model.predict(X_test)

Planting Trees...
Predicting...


In [55]:
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Not Successful', 'Successful']))

Classification Report:
                precision    recall  f1-score   support

Not Successful       0.72      0.79      0.75      1136
    Successful       0.65      0.57      0.61       796

      accuracy                           0.70      1932
     macro avg       0.69      0.68      0.68      1932
  weighted avg       0.69      0.70      0.69      1932



In [56]:
importances = rf_model.feature_importances_
feature_names = X_encoded.columns

features_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
features_df = features_df.sort_values(by='Importance', ascending=False)

features_df.head(10)

,Feature,Importance
495,Size_Varies with device,0.106985
0,Price,0.035385
33,Type_Paid,0.034380
14,Category_GAME,0.031087
20,Category_MEDICAL,0.019760
36,Content Rating_Teen,0.019514
11,Category_FAMILY,0.013671
34,Content Rating_Everyone 10+,0.012846
35,Content Rating_Mature 17+,0.012793
29,Category_TOOLS,0.011836


In [60]:
import pandas as pd
import warnings

# Suppress scikit-learn feature name warnings for a cleaner terminal output
warnings.filterwarnings('ignore', category=UserWarning) 

def predict_app_success(model, training_columns):
    print("\n" + "="*40)
    print("🚀 APP SUCCESS PREDICTOR (100k+ Installs)")
    print("="*40)
    
    # 1. Gather raw input from the console
    category = input("Enter Category (e.g., GAME, MEDICAL, BUSINESS): ").strip().upper()
    app_type = input("Enter Type (Free or Paid): ").strip().capitalize()
    price = float(input("Enter Price (e.g., 0.00 or 4.99): "))
    content_rating = input("Enter Content Rating (e.g., Everyone, Teen, Mature 17+): ").strip()
    varies = input("Does the size vary with device? (y/n): ").strip().lower()

    # 2. Create a blank template matching the exact training columns (all zeros)
    input_data = pd.DataFrame(0, index=[0], columns=training_columns)

    # 3. Inject the continuous numerical data
    if 'Price' in training_columns:
        input_data['Price'] = price

    # 4. Inject the categorical data (flip the specific 0 to a 1)
    # We use 'if column in training_columns' to handle 'drop_first=True' logic safely
    cat_col = f'Category_{category}'
    if cat_col in training_columns:
        input_data[cat_col] = 1
        
    type_col = f'Type_{app_type}'
    if type_col in training_columns:
        input_data[type_col] = 1
        
    rating_col = f'Content Rating_{content_rating}'
    if rating_col in training_columns:
        input_data[rating_col] = 1
        
    # Handle the MNAR feature you successfully identified earlier
    if varies == 'y' and 'Size_Varies with device' in training_columns:
        input_data['Size_Varies with device'] = 1

    # 5. Run the prediction
    prediction = model.predict(input_data)[0]
    probability = model.predict_proba(input_data)[0][1] # Probability of being a Hit

    # 6. Output the results cleanly to the console
    print("\n--- RESULTS ---")
    if prediction == 1:
        print(f"🔮 Prediction: HIT (High Volume Expected)")
        print(f"📊 Confidence: {probability * 100:.1f}%")
    else:
        print(f"📉 Prediction: FLOP (Niche / Low Volume)")
        print(f"📊 Confidence: {(1 - probability) * 100:.1f}%")
    print("="*40 + "\n")

# Run the interactive loop 
# (Make sure to pass your specific trained model and training columns)
while True:
    predict_app_success(rf_model, X_train.columns) 
    
    cont = input("Test another app? (y/n): ").strip().lower()
    if cont != 'y':
        print("Exiting predictor...")
        break


🚀 APP SUCCESS PREDICTOR (100k+ Installs)



--- RESULTS ---
🔮 Prediction: HIT (High Volume Expected)
📊 Confidence: 70.1%

Exiting predictor...
